# Vector Embeddings & Semantic Retrieval
## Converting Documents to Vectors for Intelligent Search

In this notebook, we'll explore the core concepts of semantic search:
- **Embeddings**: Convert text to numerical vectors that capture meaning
- **Vector Storage**: Store and organize embeddings efficiently
- **Similarity Search**: Find documents semantically similar to a query
- **Retrieval Interfaces**: Create flexible ways to access relevant documents
- **Context Integration**: Use retrieved documents in LLM prompts

This segment bridges document ingestion (Segment 2) and intelligent context routing (Segment 4).

## Step 1: Setup & Configuration

Initialize environment variables and core imports for our embedding and retrieval system.

In [10]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

## Step 2: Import Core Libraries

Import the key components for embeddings and vector storage:
- **Document**: Data structure for text content and metadata
- **OpenAIEmbeddings**: Convert text to semantic vectors using OpenAI models
- **Chroma**: Vector database for storing and querying embeddings

In [11]:
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

## Step 3: Sample Documents

We'll use three sample documents about pricing strategy (output from Segment 2's pipeline).

**Document Features:**
- **page_content**: The actual text that will be embedded
- **metadata**: Topic classification (finance vs. product) for filtering and ranking

In [12]:
documents = [
    Document(
        page_content="Revenue increased by 25% in Q3 due to enterprise adoption.",
        metadata={"source": "report.pdf", "topic": "finance"}
    ),
    Document(
        page_content="Customers prefer subscription-based pricing models.",
        metadata={"source": "notes.md", "topic": "product"}
    ),
    Document(
        page_content="AI platform launched with tiered pricing for mid-size companies.",
        metadata={"source": "product.json", "topic": "product"}
    )
]

## Step 4: Create Embeddings

Generate semantic embeddings using OpenAI's **text-embedding-3-small** model.

**Why This Model?**
- **Lightweight**: Faster inference and lower costs
- **High Quality**: Still captures semantic meaning effectively
- **Efficient**: Balances speed and accuracy for production use

**What Are Embeddings?**
Embeddings convert text into high-dimensional vectors (e.g., 1,536 dimensions) where:
- Semantically similar texts have vectors close together
- Dissimilar texts have vectors far apart
- Distance/similarity can be calculated with dot product or cosine similarity

In [13]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"   # latest recommended lightweight model
    ,openai_api_key=os.getenv("OPENAI_API_KEY")
)

## Step 5: Build Vector Store

Store embeddings in Chroma, a vector database optimized for semantic search.

**What Chroma Does:**
1. **Embeds** all documents using our embedding model
2. **Stores** the vectors with associated metadata
3. **Indexes** for fast similarity search
4. **Retrieves** most similar vectors for queries

This is the foundation for semantic search - we can now find documents by meaning, not just keywords.

In [14]:
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="context_demo"
)

## Step 6: Similarity Search (Direct Method)

Perform semantic similarity search directly on the vector store.

**How It Works:**
1. **Embed Query**: Convert the user query to a vector
2. **Calculate Distances**: Find vectors closest to the query vector
3. **Rank Results**: Return top K most similar documents
4. **Display Output**: Show content and metadata of results

This is the most straightforward way to retrieve relevant documents.

In [15]:
query = "What pricing strategy should we use for enterprise customers?"

results = vector_store.similarity_search(
    query,
    k=2
)

for r in results:
    print("----")
    print(r.page_content)
    print(r.metadata)

----
AI platform launched with tiered pricing for mid-size companies.
{'topic': 'product', 'source': 'product.json'}
----
Customers prefer subscription-based pricing models.
{'topic': 'product', 'source': 'notes.md'}


## Step 7: Retriever Interface (Flexible Method)

Convert the vector store into a retriever - a more flexible interface for document access.

**Why Use a Retriever?**
- **Standard Interface**: Works with LangChain chains and agents
- **Composable**: Can be combined with other retrieval methods
- **Configurable**: Easy to adjust search parameters (k, search type)
- **Chainable**: Can be used directly in LLM prompt construction

The retriever pattern enables building complex systems without tightly coupling to specific vector stores.

In [16]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

docs = retriever.invoke(query)

for d in docs:
    print(d.page_content)

AI platform launched with tiered pricing for mid-size companies.
Customers prefer subscription-based pricing models.
Revenue increased by 25% in Q3 due to enterprise adoption.


In [17]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(model="gpt-4o", temperature=0)

context = "\n\n".join([d.page_content for d in docs])

messages = [
    SystemMessage(content="You are a product strategist."),
    HumanMessage(content=f"""
Context:
{context}

Question:
{query}
""")
]

response = llm.invoke(messages)

print(response.content)

Given the context and the information provided, here are some strategic considerations for pricing your AI platform for enterprise customers:

1. **Value-Based Pricing**: Since enterprise customers have already contributed to a 25% revenue increase, it's clear they see significant value in your platform. Consider a value-based pricing strategy where prices are set based on the perceived value to the customer. This approach can maximize revenue by aligning the price with the benefits and ROI the enterprise customers receive.

2. **Tiered Pricing with Customization**: While you already have tiered pricing for mid-size companies, consider offering a more flexible, customizable tier for enterprise customers. This could include additional features, dedicated support, or integration services that are tailored to the specific needs of larger organizations.

3. **Subscription-Based Model**: Since customers prefer subscription-based pricing models, continue to offer this structure for enterpris

## Step 8: Context-Augmented Generation

Now we complete the RAG loop by using retrieved documents as context for the LLM.

**The RAG Pattern:**
```
Query
   ↓
[Retrieve] → Get semantically similar documents
   ↓
[Contextualize] → Combine documents into context string
   ↓
[Generate] → Use context in LLM prompt
   ↓
Answer grounded in retrieved information
```

This demonstrates the power of combining:
- **Semantic Search** (find relevant documents)
- **Context Engineering** (structure the prompt)
- **LLM Generation** (produce informed responses)

## Key Takeaways

### The Semantic Search Foundation

Embeddings and vector storage enable meaning-based document retrieval:

```
Text Documents
    ↓
[Embedding Model]
    ↓
High-Dimensional Vectors
    ↓
[Vector Database]
    ↓
Fast Similarity Search
    ↓
Semantic Results
```

### Why Embeddings Matter

1. **Beyond Keywords**: Find documents by meaning, not exact word matches
2. **Semantic Understanding**: Captures relationships between concepts
3. **Fast Retrieval**: Vector operations are highly optimized
4. **Flexible Combinations**: Can combine with metadata filtering and ranking

### Practical Patterns

**Direct Search:**
```python
results = vector_store.similarity_search(query, k=2)
```

**Retriever Interface:**
```python
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
docs = retriever.invoke(query)
```

### What's Next (Segment 4)

With semantic retrieval working, we can now:
- **Rank** results by multiple factors (similarity + priority + recency)
- **Route** queries to appropriate context sources
- **Resolve** conflicts between competing documents
- **Build** intelligent context selection systems